# 🧬 Ensemble Model — Oral Cancer Detection
### EfficientNetB0 + DenseNet121 + ResNet50 → Weighted Average + Meta-Learner

**Pipeline overview:**
1. Load all 3 saved models
2. Rebuild the exact val & test sets (same `random_state=42` splits)
3. Generate predictions from each model with their correct preprocessing
4. Method A — Weighted Average (no extra training)
5. Method B — Logistic Regression Meta-Learner (trained on val predictions)
6. Method C — Keras Meta-Head (small trainable Dense layer on top of frozen models)
7. Full evaluation + confusion matrix + ROC curves

## 0. Imports & Config

In [ ]:
!pip install albumentations -q

import os, cv2, numpy as np, pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_curve, roc_auc_score)
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, Dropout, Concatenate,
                                      Input, BatchNormalization)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
import kagglehub

# ── Constants (must match your individual training notebooks) ──
TARGET_SIZE = 224
BATCH_SIZE  = 32
SEED        = 42

# ── Paths to your saved models ────────────────────────────────
# Update these to wherever your .keras files were saved
MODEL_PATHS = {
    'EfficientNetB0': '/content/best_EfficientNetB0.keras',
    'DenseNet121'   : '/content/best_DenseNet121.keras',
    'ResNet50'      : '/content/best_ResNet50.keras',
}

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 1. Recreate the Datasets
> Same logic as individual notebooks — same random seed → identical splits

In [ ]:
# ── Download datasets (skip if already on disk) ───────────────
path_big   = kagglehub.dataset_download('zaidpy/oral-cancer-dataset')
path_small = kagglehub.dataset_download('shivam17299/oral-cancer-lips-and-tongue-images')

# ── Build df_train (700 cancer / 700 non-cancer) ─────────────
datasets = [
    {
        'root'      : os.path.join(path_big, 'Oral Cancer/Oral Cancer Dataset'),
        'subfolders': {'CANCER': 'cancer', 'NON CANCER': 'non_cancer'}
    },
    {
        'root'      : os.path.join(path_big, 'Oral cancer Dataset 2.0/OC Dataset kaggle new'),
        'subfolders': {'CANCER': 'cancer', 'NON CANCER': 'non_cancer'}
    }
]

filepaths, labels = [], []
for ds in datasets:
    for folder_name, label in ds['subfolders'].items():
        folder_path = os.path.join(ds['root'], folder_name)
        if os.path.exists(folder_path):
            for img in os.listdir(folder_path):
                if img.lower().endswith(('.png', '.jpg', '.jpeg')):
                    filepaths.append(os.path.join(folder_path, img))
                    labels.append(label)

df_train = pd.DataFrame({'filename': filepaths, 'label': labels})
df_train = df_train.drop_duplicates(subset='filename').reset_index(drop=True)
df_train = df_train.sample(frac=1, random_state=SEED).reset_index(drop=True)

cancer_df    = df_train[df_train['label'] == 'cancer']
noncancer_df = df_train[df_train['label'] == 'non_cancer']
cancer_df    = cancer_df.sample(n=700, random_state=SEED)
df_train     = pd.concat([cancer_df, noncancer_df]).reset_index(drop=True)
df_train     = df_train.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f'Training pool (balanced): {len(df_train)}')
print(df_train['label'].value_counts())

# ── Build df_test ─────────────────────────────────────────────
filepaths_test, labels_test = [], []
label_map_small = {'cancer': 'cancer', 'non-cancer': 'non_cancer'}

for folder_name, label in label_map_small.items():
    folder_path = os.path.join(path_small, 'OralCancer', folder_name)
    if os.path.exists(folder_path):
        for img in os.listdir(folder_path):
            if img.lower().endswith(('.png', '.jpg', '.jpeg')):
                filepaths_test.append(os.path.join(folder_path, img))
                labels_test.append(label)

df_test = pd.DataFrame({'filename': filepaths_test, 'label': labels_test})
df_test = df_test.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f'\nTest set: {len(df_test)}')
print(df_test['label'].value_counts())

## 2. Preprocessing Helpers
> **Critical:** each model was trained with its own preprocessing. We must apply the right one at inference time.

In [ ]:
from tensorflow.keras.applications.resnet50       import preprocess_input as resnet50_preprocess
from tensorflow.keras.applications.efficientnet   import preprocess_input as efficientnet_preprocess
from tensorflow.keras.applications.densenet       import preprocess_input as densenet_preprocess

# ── Shared image-cleaning steps (same as training) ────────────
def apply_clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_enhanced = clahe.apply(l)
    return cv2.cvtColor(cv2.merge([l_enhanced, a, b]), cv2.COLOR_LAB2RGB)

def smart_resize(img, target_size=TARGET_SIZE):
    h, w = img.shape[:2]
    min_dim = min(h, w)
    start_h = (h - min_dim) // 2
    start_w = (w - min_dim) // 2
    img_cropped = img[start_h:start_h+min_dim, start_w:start_w+min_dim]
    return cv2.resize(img_cropped, (target_size, target_size))

def sharpen(img):
    gaussian  = cv2.GaussianBlur(img, (0, 0), 2.0)
    sharpened = cv2.addWeighted(img, 1.5, gaussian, -0.5, 0)
    return np.clip(sharpened, 0, 255).astype(np.uint8)

def load_and_preprocess(filepath, preprocess_fn, target_size=TARGET_SIZE):
    """Load one image, apply shared cleaning + model-specific normalisation."""
    img = cv2.imread(filepath)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = apply_clahe(img)
    img = sharpen(img)
    img = smart_resize(img, target_size)
    img = img.astype('float32')
    img = preprocess_fn(img)     # model-specific normalisation
    return img

# ── Map model name → its preprocessing function ───────────────
PREPROCESS_FN = {
    'EfficientNetB0': efficientnet_preprocess,
    'DenseNet121'   : densenet_preprocess,
    'ResNet50'      : resnet50_preprocess,
}

print('✅ Preprocessing helpers ready')
print('Each model will receive images normalised with its own preprocess_input.')

## 3. Rebuild the Val Split & Load Test Images
> We replicate the exact 80/20 stratified split from training (random_state=42).

In [ ]:
# ── Val split — using preprocessed folder (ResNet50 path as reference) ─
# If you stored preprocessed images in per-model folders, point to any one;
# the filenames and labels are the same across all three.
PREPROCESS_DIR = '/content/preprocessed_resnet18'   # adjust if different

all_paths, all_labels = [], []
for label in ['cancer', 'non_cancer']:
    folder = os.path.join(PREPROCESS_DIR, label)
    if os.path.exists(folder):
        for fname in os.listdir(folder):
            all_paths.append(os.path.join(folder, fname))
            all_labels.append(1 if label == 'cancer' else 0)
    else:
        # fallback: derive paths from df_train filenames
        pass

_, val_paths, _, y_val = train_test_split(
    all_paths, all_labels,
    test_size=0.2, random_state=SEED, stratify=all_labels
)

y_val = np.array(y_val)
print(f'Val  : {len(val_paths)} images  '
      f'(cancer={sum(y_val==1)}, non_cancer={sum(y_val==0)})')

# ── Test labels ───────────────────────────────────────────────
y_test = np.array([1 if r['label'] == 'cancer' else 0
                   for _, r in df_test.iterrows()])
test_paths = list(df_test['filename'])
print(f'Test : {len(test_paths)} images  '
      f'(cancer={sum(y_test==1)}, non_cancer={sum(y_test==0)})')

## 4. Load Models & Generate Per-Model Predictions

In [ ]:
def load_images(paths, preprocess_fn):
    """Load a list of image paths into a float32 numpy array."""
    images = []
    valid  = []
    for p in tqdm(paths, desc='  loading'):
        img = load_and_preprocess(p, preprocess_fn)
        if img is not None:
            images.append(img)
            valid.append(True)
        else:
            valid.append(False)
    return np.array(images), np.array(valid)


# ── Storage for predictions ───────────────────────────────────
val_preds  = {}   # {model_name: np.array shape (N,)}
test_preds = {}
models     = {}

for name, path in MODEL_PATHS.items():
    print(f'\n{'='*55}')
    print(f'  Loading {name}  →  {path}')
    print(f'{'='*55}')

    model = tf.keras.models.load_model(path)
    model.trainable = False
    models[name] = model

    preprocess_fn = PREPROCESS_FN[name]

    # Val predictions
    print('  Preprocessing val set...')
    X_val_m, _ = load_images(val_paths, preprocess_fn)
    val_preds[name] = model.predict(X_val_m, batch_size=BATCH_SIZE,
                                    verbose=1).flatten()

    # Test predictions
    print('  Preprocessing test set...')
    X_test_m, _ = load_images(test_paths, preprocess_fn)
    test_preds[name] = model.predict(X_test_m, batch_size=BATCH_SIZE,
                                     verbose=1).flatten()

    auc = roc_auc_score(y_val, val_preds[name])
    print(f'  ✅ Val AUC: {auc:.4f}')

print('\n✅ All 3 models loaded and predictions cached.')

## 5. Evaluation Helper

In [ ]:
def evaluate(y_true, y_prob, name='Model', split='Test'):
    """Print metrics and plot confusion matrix + ROC."""
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    auc_score   = roc_auc_score(y_true, y_prob)
    best_idx    = np.argmax(tpr - fpr)
    best_thresh = thresholds[best_idx]

    y_pred_def = (y_prob >= 0.50).astype(int)
    y_pred_opt = (y_prob >= best_thresh).astype(int)

    print(f'\n{"="*55}')
    print(f'  {name}  |  {split} Set')
    print(f'  AUC: {auc_score:.4f}   Optimal threshold: {best_thresh:.4f}')
    print(f'{'='*55}')
    print(classification_report(y_true, y_pred_opt,
                                target_names=['Non-Cancer', 'Cancer']))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred_opt)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=['Non-Cancer', 'Cancer'],
                yticklabels=['Non-Cancer', 'Cancer'])
    axes[0].set_title(f'{name} — Confusion Matrix ({split})')
    axes[0].set_ylabel('Actual')
    axes[0].set_xlabel('Predicted')

    # ROC
    axes[1].plot(fpr, tpr, lw=2, label=f'AUC = {auc_score:.3f}')
    axes[1].scatter(fpr[best_idx], tpr[best_idx], s=100, color='red', zorder=5,
                    label=f'Threshold = {best_thresh:.3f}')
    axes[1].plot([0,1],[0,1],'k--',lw=1)
    axes[1].set_title(f'{name} — ROC Curve ({split})')
    axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{name}_{split}_evaluation.png', dpi=150)
    plt.show()

    return auc_score, best_thresh

print('✅ evaluate() helper ready')

## 6. Individual Model Baselines

In [ ]:
individual_aucs = {}
for name in MODEL_PATHS:
    auc, _ = evaluate(y_test, test_preds[name], name=name, split='Test')
    individual_aucs[name] = auc

print('\nIndividual Test AUCs:')
for k, v in individual_aucs.items():
    print(f'  {k}: {v:.4f}')

## 7. Method A — Weighted Average Ensemble
> Weights derived from each model's **val AUC** (no extra training needed).

In [ ]:
# ── Compute val AUCs ──────────────────────────────────────────
val_aucs = {name: roc_auc_score(y_val, val_preds[name])
            for name in MODEL_PATHS}

print('Val AUCs:')
for k, v in val_aucs.items():
    print(f'  {k}: {v:.4f}')

# ── Normalise to get weights that sum to 1 ───────────────────
total_auc  = sum(val_aucs.values())
weights    = {name: auc / total_auc for name, auc in val_aucs.items()}

print('\nDerived weights (proportional to val AUC):')
for k, v in weights.items():
    print(f'  {k}: {v:.4f}')

# ── Weighted prediction ───────────────────────────────────────
weighted_val_prob  = sum(weights[n] * val_preds[n]  for n in MODEL_PATHS)
weighted_test_prob = sum(weights[n] * test_preds[n] for n in MODEL_PATHS)

# Evaluate
evaluate(y_val,  weighted_val_prob,  name='WeightedEnsemble', split='Val')
evaluate(y_test, weighted_test_prob, name='WeightedEnsemble', split='Test')

## 8. Method B — Logistic Regression Meta-Learner
> Stacks the 3 model probability outputs as features and trains a tiny LR model on the **val set**.

In [ ]:
# ── Stack val predictions as (N, 3) feature matrix ───────────
X_meta_val  = np.column_stack([val_preds[n]  for n in MODEL_PATHS])
X_meta_test = np.column_stack([test_preds[n] for n in MODEL_PATHS])

print('Meta-feature shapes:')
print(f'  Val  X: {X_meta_val.shape}   y: {y_val.shape}')
print(f'  Test X: {X_meta_test.shape}  y: {y_test.shape}')

# ── Train meta-learner on val predictions ────────────────────
# NOTE: For a more rigorous setup, use cross-validated out-of-fold
# predictions from the training set as meta-features. Here we use
# val set because that's the held-out data you have.
meta_lr = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
meta_lr.fit(X_meta_val, y_val)

print('\nMeta-learner coefficients (model weights learned):')
for name, coef in zip(MODEL_PATHS.keys(), meta_lr.coef_[0]):
    print(f'  {name}: {coef:.4f}')

# ── Predict on test ───────────────────────────────────────────
meta_test_prob = meta_lr.predict_proba(X_meta_test)[:, 1]
evaluate(y_test, meta_test_prob, name='LR_MetaLearner', split='Test')

## 9. Method C — Keras Meta-Head (Trainable Fusion)
> Freeze all 3 models. Add a tiny Dense(16) → Dense(1) head. Train only the head for ~15 epochs.

In [ ]:
# ── Custom early-stop callback (mirrors your individual notebooks) ─
class CombinedMetricCallback(tf.keras.callbacks.Callback):
    def __init__(self, patience=10, model_name='ensemble'):
        super().__init__()
        self.patience     = patience
        self.model_name   = model_name
        self.best_score   = 0
        self.wait         = 0
        self.best_weights = None

    def on_epoch_end(self, epoch, logs=None):
        auc      = logs.get('val_auc', 0)
        acc      = logs.get('val_accuracy', 0)
        combined = 0.6 * auc + 0.4 * acc
        print(f'\n  📊 val_auc={auc:.4f} | val_acc={acc:.4f} '
              f'| combined={combined:.4f} (best={self.best_score:.4f})')
        if combined > self.best_score + 0.001:
            self.best_score   = combined
            self.wait         = 0
            self.best_weights = self.model.get_weights()
            self.model.save(f'best_{self.model_name}.keras')
            print('  ✅ Improved → saved')
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.model.set_weights(self.best_weights)
                self.model.stop_training = True
                print(f'\n🛑 Early stopping at epoch {epoch+1}')

    def on_train_end(self, logs=None):
        if self.best_weights:
            self.model.set_weights(self.best_weights)
            print(f'\n✅ Best weights restored (score={self.best_score:.4f})')


print('✅ CombinedMetricCallback defined')

In [ ]:
# ── Build the Keras ensemble model ───────────────────────────
# Strategy: shared input → 3 frozen sub-models → Concatenate → meta-head
# Each sub-model uses its own preprocessing inside the pipeline,
# BUT since Keras models share the same input tensor here, we run
# each model on its own pre-processed arrays (Method C-alt below).
#
# The cleanest approach for mixed preprocessing:
# Use 3 SEPARATE inputs, one per model, each pre-processed externally.

inp_eff   = Input(shape=(TARGET_SIZE, TARGET_SIZE, 3), name='input_efficientnet')
inp_dense = Input(shape=(TARGET_SIZE, TARGET_SIZE, 3), name='input_densenet')
inp_res   = Input(shape=(TARGET_SIZE, TARGET_SIZE, 3), name='input_resnet')

# Freeze the loaded models
for name, m in models.items():
    m.trainable = False

# Get outputs (shape: batch × 1 each)
out_eff   = models['EfficientNetB0'](inp_eff)
out_dense = models['DenseNet121'](inp_dense)
out_res   = models['ResNet50'](inp_res)

# Concatenate → (batch, 3)
concat = Concatenate(name='concat_probs')([out_eff, out_dense, out_res])

# Tiny trainable meta-head
x = Dense(16, activation='relu', name='meta_dense1')(concat)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)
output = Dense(1, activation='sigmoid', name='meta_output')(x)

ensemble_model = Model(
    inputs=[inp_eff, inp_dense, inp_res],
    outputs=output,
    name='EnsembleMetaHead'
)

ensemble_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

trainable_params = sum(np.prod(v.shape) for v in ensemble_model.trainable_variables)
print(f'Trainable params (meta-head only): {trainable_params:,}')
ensemble_model.summary(line_length=80)

In [ ]:
# ── Prepare multi-input arrays ────────────────────────────────
# We already have val_preds from individual model inference.
# For the Keras model we need the raw preprocessed images per model.

print('Loading val images for each model (3× preprocessing)...')
X_val_dict  = {}
X_test_dict = {}

for name in MODEL_PATHS:
    fn = PREPROCESS_FN[name]
    print(f'  {name} — val')
    X_val_dict[name],  _ = load_images(val_paths,  fn)
    print(f'  {name} — test')
    X_test_dict[name], _ = load_images(test_paths, fn)

print('✅ Done')

In [ ]:
# ── Train the meta-head on the val set ───────────────────────
# (Use the val set as both train & internal-val for the meta-head.
#  This is acceptable because the 3 base models never saw val during training.)

X_eff_val   = X_val_dict['EfficientNetB0']
X_dense_val = X_val_dict['DenseNet121']
X_res_val   = X_val_dict['ResNet50']

# 80/20 internal split of val set for meta-head training
n = len(y_val)
split = int(0.8 * n)
idx   = np.random.RandomState(SEED).permutation(n)
tr, vl = idx[:split], idx[split:]

train_inputs = [X_eff_val[tr], X_dense_val[tr], X_res_val[tr]]
val_inputs   = [X_eff_val[vl], X_dense_val[vl], X_res_val[vl]]

history = ensemble_model.fit(
    train_inputs, y_val[tr],
    validation_data=(val_inputs, y_val[vl]),
    epochs=30,
    batch_size=16,
    callbacks=[
        CombinedMetricCallback(patience=10, model_name='EnsembleMetaHead'),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=4, min_lr=1e-7, verbose=1)
    ],
    verbose=1
)
print('\n✅ Meta-head training complete')

In [ ]:
# ── Training history plot ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, key, title in zip(axes,
                           ['accuracy', 'loss', 'auc'],
                           ['Accuracy', 'Loss', 'AUC']):
    ax.plot(history.history[key],     label='Train')
    ax.plot(history.history[f'val_{key}'], label='Val')
    ax.set_title(f'Meta-Head — {title}')
    ax.legend(); ax.grid(alpha=0.3)
plt.suptitle('Ensemble Meta-Head Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('ensemble_metahead_history.png', dpi=150)
plt.show()

In [ ]:
# ── Test-set evaluation of the Keras meta-head ───────────────
ensemble_test_prob = ensemble_model.predict(
    [X_test_dict['EfficientNetB0'],
     X_test_dict['DenseNet121'],
     X_test_dict['ResNet50']],
    batch_size=BATCH_SIZE, verbose=1
).flatten()

evaluate(y_test, ensemble_test_prob,
         name='KerasMetaHead_Ensemble', split='Test')

## 10. Head-to-Head Comparison

In [ ]:
results = {}

# Individual models
for name in MODEL_PATHS:
    results[name] = roc_auc_score(y_test, test_preds[name])

# Ensembles
results['Weighted Avg']     = roc_auc_score(y_test, weighted_test_prob)
results['LR Meta-Learner']  = roc_auc_score(y_test, meta_test_prob)
results['Keras Meta-Head']  = roc_auc_score(y_test, ensemble_test_prob)

print('\n🏆 Test AUC Comparison')
print('─' * 40)
for name, auc in sorted(results.items(), key=lambda x: -x[1]):
    tag = '⭐ ENSEMBLE' if 'Meta' in name or 'Avg' in name else ''
    print(f'  {name:<22}: {auc:.4f}  {tag}')

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
colors  = ['#4C72B0']*3 + ['#55A868', '#C44E52', '#8172B2']
names   = list(results.keys())
aucs    = list(results.values())

bars = ax.barh(names, aucs, color=colors, edgecolor='white', height=0.6)
ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=10)
ax.set_xlim(min(aucs)*0.97, 1.02)
ax.set_xlabel('AUC Score')
ax.set_title('Oral Cancer Detection — Model Comparison (Test AUC)',
             fontsize=13, fontweight='bold')
ax.axvline(x=max(aucs), color='red', linestyle='--', alpha=0.5, label='Best')
ax.legend(); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('model_comparison_auc.png', dpi=150)
plt.show()

## 11. Save Ensemble & Export Predictions

In [ ]:
# ── Save the best ensemble model ─────────────────────────────
ensemble_model.save('/content/best_EnsembleMetaHead.keras')
print('✅ Keras meta-head ensemble saved → /content/best_EnsembleMetaHead.keras')

# ── Export test predictions to CSV ───────────────────────────
fpr, tpr, thresholds = roc_curve(y_test, ensemble_test_prob)
best_thresh = thresholds[np.argmax(tpr - fpr)]

export_df = pd.DataFrame({
    'filepath'             : test_paths,
    'actual'               : ['cancer' if y==1 else 'non_cancer' for y in y_test],
    'prob_efficientnet'    : np.round(test_preds['EfficientNetB0'], 4),
    'prob_densenet'        : np.round(test_preds['DenseNet121'], 4),
    'prob_resnet'          : np.round(test_preds['ResNet50'], 4),
    'prob_weighted_avg'    : np.round(weighted_test_prob, 4),
    'prob_lr_meta'         : np.round(meta_test_prob, 4),
    'prob_keras_ensemble'  : np.round(ensemble_test_prob, 4),
    'pred_ensemble_default': ['cancer' if p>=0.5 else 'non_cancer'
                              for p in ensemble_test_prob],
    'pred_ensemble_optimal': ['cancer' if p>=best_thresh else 'non_cancer'
                              for p in ensemble_test_prob],
    'correct_default'      : (ensemble_test_prob >= 0.50).astype(int) == y_test,
    'correct_optimal'      : (ensemble_test_prob >= best_thresh).astype(int) == y_test,
})

export_df.to_csv('/content/ensemble_test_predictions.csv', index=False)
print('✅ Predictions saved → /content/ensemble_test_predictions.csv')
print(f'   Optimal threshold: {best_thresh:.4f}')
print(f'   Accuracy (optimal): {export_df["correct_optimal"].mean()*100:.1f}%')